In [0]:
# ---------------------------------------------------------
# 0. Run Imports 
# ---------------------------------------------------------

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import hashlib


In [0]:
# ---------------------------------------------------------
# 1. Split Records into Valid and Invalid Dataframes
# ---------------------------------------------------------
def split_valid_invalid_records(df):

    invalid_df = (
        df
        .filter(F.col("_corrupt_record").isNotNull())
    )

    valid_df = (
        df
        .filter(F.col("_corrupt_record").isNull())
        .drop("_corrupt_record")
    )

    return valid_df, invalid_df

def build_quarantine_dataframe(
    invalid_df,
    source_system,
    bronze_table,
    file_path,
    batch_id,
    pipeline_run_id,
    file_hash,
    file_size
):

    quarantine_df = (
        invalid_df
        .withColumn(
            "source_system",
            F.lit(source_system)
        )
        .withColumn(
            "bronze_table",
            F.lit(bronze_table)
        )
        .withColumn(
            "file_path",
            F.lit(file_path)
        )
        .withColumn(
            "file_name",
            F.element_at(
                F.split(F.lit(file_path), "/"),
                -1
            )
        )
        .withColumn(
            "batch_id",
            F.lit(batch_id)
        )
        .withColumn(
            "pipeline_run_id",
            F.lit(pipeline_run_id)
        )
        .withColumn(
            "quarantine_reason",
            F.lit("CORRUPT_RECORD")
        )
        .withColumn(
            "validation_rule",
            F.lit("SPARK_PARSING_ERROR")
        )
        .withColumn(
            "raw_record",
            F.col("_corrupt_record")
        )
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "file_hash",
            F.lit(file_hash)
        )
        .withColumn(
            "file_size",
            F.lit(file_size)
        )
        .select(
            "source_system",
            "bronze_table",
            "file_path",
            "file_name",
            "batch_id",
            "pipeline_run_id",
            "quarantine_reason",
            "validation_rule",
            "raw_record",
            "ingestion_timestamp",
            "file_hash",
            "file_size"
        )
    )

    return quarantine_df

In [0]:
# ---------------------------------------------------------
# 2. Save Invalid Records to Quarantine Table
# ---------------------------------------------------------

def write_to_quarantine_table(
    quarantine_df,
    quarantine_table
):

    (
        quarantine_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(quarantine_table)
    )